# Entrainement des modeles sur GPU (Google Colab)

**Le poste de travail du projet n'a pas de GPU.** Les entrainements
menes en local (3 epoques, 320 px) ne sont que des tests de chaine :
ils prouvent que le code fonctionne, pas que les modeles sont bons.
Le vrai entrainement se fait ici.

## Une seule chose a faire avant de lancer

Menu **Execution -> Modifier le type d'execution -> GPU (T4)**.

Ensuite *Execution -> Tout executer*. Il n'y a rien a televerser : le
projet est clone depuis GitHub et les datasets sont regeneres sur place.

| Plateforme | Session max | Convoyeur, 200 epoques a 1024 px |
|---|---|---|
| Colab T4 | quelques heures | 3 a 4 h : **risque de coupure** |
| Kaggle P100 | 12 h | 2 h 30 : preferable |

Ce notebook utilise `--epochs 120` pour tenir dans une session Colab.


## 1. Verifier le GPU

L'assertion arrete le notebook si l'accelerateur n'a pas ete active.
Sans elle, l'entrainement demarrerait sur processeur et tournerait des
heures pour rien.


In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN GPU')
assert torch.cuda.is_available(), (
    'Activez le GPU : Execution -> Modifier le type d execution -> GPU (T4)')


## 2. Cloner le projet et installer


In [ ]:
!git clone -q https://github.com/othmanedhilou/MODELE.git
%cd MODELE
!pip install -q ultralytics
import ultralytics; print('ultralytics', ultralytics.__version__)


## 3. Generer les datasets

Les images synthetiques ne sont pas versionnees : elles se regenerent
a l'identique en une minute grace au parametre `--seed`. Televerser un
giga d'images reproductibles n'aurait aucun interet.

Le generateur convoyeur produit les **9 classes localisables** de la
taxonomie. Le desalignement est traite par la couche vision classique :
c'est une propriete globale de l'image, pas un objet a encadrer.


In [ ]:
!python scripts/generer_dataset_convoyeur.py --nombre 1200
!python scripts/generer_dataset_eclairage.py --nombre 800


### Optionnel mais nettement meilleur : incruster sur vos images reelles

Le fond, l'eclairage, la matiere et le bruit du capteur sont alors
reels ; seul le defaut est dessine. Deposez vos images de bande, meme
sans aucun defaut, dans `data/frames/convoyeur`.


In [ ]:
# !python scripts/generer_dechirures_sur_reel.py \
#        --source data/frames/convoyeur --par-image 3


## 4. Controler le dataset

Trois minutes ici evitent d'attendre trois heures un resultat fausse
par une fuite train/val ou une classe vide.


In [ ]:
!python -m src.prepare.check_dataset --modele convoyeur
!python -m src.mlops.registre --empreinte convoyeur


## 5. Entrainer le modele convoyeur

En cas de `CUDA out of memory`, relancez avec `--batch 4`, puis 2.


In [ ]:
!python -m src.train.train --modele convoyeur --epochs 120


## 6. Evaluer sur le lot de test

Le lot de test n'a jamais ete vu pendant l'entrainement : c'est le
seul chiffre presentable comme performance reelle dans le rapport.

**Regardez le detail par classe, pas seulement le mAP global.** Une
moyenne correcte peut cacher deux classes a zero : lors du test local,
`bord_effiloche` et `deversement` etaient a 0, parce que ce sont les
deux classes situees en bordure ou hors de la bande, la ou une
resolution reduite ecrase le signal.


In [ ]:
!python -m src.train.evaluer --modele convoyeur --exporter onnx


### Courbes et matrice de confusion pour le rapport


In [ ]:
from IPython.display import Image, display
import glob
for chemin in sorted(glob.glob('runs/convoyeur/train/*.png')):
    print(chemin)
    display(Image(chemin, width=760))


## 7. Les autres modeles

Le modele vehicules exige un dataset : il n'a pas de generateur
synthetique, une voiture dessinee ne se transfere pas au reel. Voir
`docs/sans_donnees.md` pour les datasets publics et l'importateur.


In [ ]:
!python -m src.train.train --modele eclairage
!python -m src.train.evaluer --modele eclairage


In [ ]:
# !python -m src.train.train --modele vehicules


## 8. Promouvoir en production

La promotion **verifie** que le modele correspond aux classes
declarees dans `configs/data_convoyeur.yaml`, et refuse sinon. Un
modele decale ne plante pas : il renvoie de mauvais noms de defauts,
ce qui est pire qu'une erreur franche.


In [ ]:
!python -m src.mlops.registre --lister
!python -m src.mlops.registre --promouvoir convoyeur --version v1
!python -m src.mlops.registre --promouvoir eclairage --version v1


## 9. Telecharger les poids

A faire **avant** la fin de la session : tout est perdu a la coupure.


In [ ]:
import shutil, os
os.makedirs('/content/resultats', exist_ok=True)
for modele in ('convoyeur', 'eclairage', 'vehicules'):
    source = f'models/{modele}/production.pt'
    if os.path.exists(source):
        shutil.copy(source, f'/content/resultats/{modele}_production.pt')
        print('pret :', modele)
shutil.copy('models/registre.json', '/content/resultats/registre.json')
shutil.make_archive('/content/resultats', 'zip', '/content/resultats')

from google.colab import files
files.download('/content/resultats.zip')


---
## De retour sur le poste local

Decompressez `resultats.zip`, placez chaque `*_production.pt` dans
`models/<modele>/production.pt`, puis :

```bash
python -m src.mlops.registre --lister
python -m src.pipeline.run_stream --camera cam_convoyeur_01
```

Le pipeline ne charge que les poids promus, et refuse ceux dont les
classes ne correspondent plus a la configuration.
